<a href="https://colab.research.google.com/github/roxanapodean/Disertatie/blob/master/Random_Forest_Multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

from google.colab import files

uploaded = files.upload()

filename = "mult25%EEC1_fuzzy100%TCO1_Inputs.asc"

with open(filename, "r") as f:
    lines = f.readlines()

lines = [line.strip() for line in lines if line.strip() != ""]

rows = []
outputs = []
max_inputs = 0

for line in lines:

    values = [float(x.strip()) for x in line.split(",")]

    output = int(values[-1])
    input_row = values[:-1]

    rows.append(input_row)
    outputs.append(output)

    max_inputs = max(max_inputs, len(input_row))

X = np.zeros((len(rows), max_inputs))

for i, row in enumerate(rows):
    X[i, :len(row)] = row

y = np.array(outputs).astype(int)

print(X.shape)
print(y.shape)
print(np.unique(y, return_counts=True))

Saving mult25%EEC1_fuzzy100%TCO1_Inputs.asc to mult25%EEC1_fuzzy100%TCO1_Inputs (1).asc
(122230, 6)
(122230,)
(array([0, 1, 2]), array([81449, 27153, 13628]))


In [ ]:
n = len(X)

train_end = int(0.30 * n)
val_end = int(0.40 * n)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=0,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

In [ ]:
C = confusion_matrix(y_test, y_pred, labels=[0,1,2])

print("Confusion matrix:")
print(C)

overall_accuracy = np.trace(C) / np.sum(C) * 100

print(f"Overall Accuracy: {overall_accuracy:.2f}%")

Confusion matrix:
[[48849    58    19]
 [    5 16307     0]
 [  136     0  7964]]
Overall Accuracy: 99.70%


In [ ]:
class_names = ["Legitim", "Fuzzy", "Multiplication"]

num_classes = 3
total = np.sum(C)

for k in range(num_classes):

    TP = C[k, k]

    FP = np.sum(C[:, k]) - TP

    FN = np.sum(C[k, :]) - TP

    TN = total - TP - FP - FN

    TPR = TP / (TP + FN) * 100 if (TP + FN) != 0 else 0
    TNR = TN / (TN + FP) * 100 if (TN + FP) != 0 else 0

    FNR = FN / (TP + FN) * 100 if (TP + FN) != 0 else 0
    FPR = FP / (TN + FP) * 100 if (TN + FP) != 0 else 0

    accuracy = (TP + TN) / (TP + TN + FP + FN) * 100

    precision = TP / (TP + FP) * 100 if (TP + FP) != 0 else 0

    f1_score = (2 * TP) / (2 * TP + FP + FN) * 100 \
               if (2 * TP + FP + FN) != 0 else 0

    print(f"\nClass {k} - {class_names[k]}")
    print(f"TP: {TP}")
    print(f"TN: {TN}")
    print(f"FP: {FP}")
    print(f"FN: {FN}")

    print(f"TPR: {TPR:.2f}")
    print(f"TNR: {TNR:.2f}")
    print(f"FPR: {FPR:.2f}")
    print(f"FNR: {FNR:.2f}")

    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Precision: {precision:.2f}%")
    print(f"F1Score: {f1_score:.2f}%")


Class 0 - Legitim
TP: 48849
TN: 24271
FP: 141
FN: 77
TPR: 99.84
TNR: 99.42
FPR: 0.58
FNR: 0.16
Accuracy: 99.70%
Precision: 99.71%
F1Score: 99.78%

Class 1 - Fuzzy
TP: 16307
TN: 56968
FP: 58
FN: 5
TPR: 99.97
TNR: 99.90
FPR: 0.10
FNR: 0.03
Accuracy: 99.91%
Precision: 99.65%
F1Score: 99.81%

Class 2 - Multiplication
TP: 7964
TN: 65219
FP: 19
FN: 136
TPR: 98.32
TNR: 99.97
FPR: 0.03
FNR: 1.68
Accuracy: 99.79%
Precision: 99.76%
F1Score: 99.04%
